
# Comparison of AI Models

## Problem
Financial illiteracy and the risk of over-indebtedness in low-income populations.

## Objetive
Evaluate and compare two artificial intelligence models to determine which generates clearer, more accurate, and more practical financial explanations.

## Models
1. Qwen2.5-0.5B-Instruct
2. MNN-Pocket-0.3B


In [1]:
# Instalación de la biblioteca transformers y torch
%pip install transformers
%pip install torch
%pip install tf-keras

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:

%pip install transformers torch pandas numpy accelerate sentencepiece safetensors


Note: you may need to restart the kernel to use updated packages.


In [3]:

import pandas as pd

# Create fictional financial cases
cases = [
    {"id": 1, "income": 450, "expenses": 280, "debt_payments": 120},
    {"id": 2, "income": 500, "expenses": 350, "debt_payments": 200},
    {"id": 3, "income": 600, "expenses": 300, "debt_payments": 100},
    {"id": 4, "income": 400, "expenses": 310, "debt_payments": 150},
    {"id": 5, "income": 700, "expenses": 400, "debt_payments": 250}
]

# Convert the cases into a DataFrame
df = pd.DataFrame(cases)

# Calculate the remaining balance
df["remaining_balance"] = (
    df["income"] - df["expenses"] - df["debt_payments"]
)

# Calculate the percentage of income used for debt payments
df["debt_percentage"] = (
    df["debt_payments"] / df["income"] * 100
)

# Display the financial data
display(df)


,id,income,expenses,debt_payments,remaining_balance,debt_percentage
0,1,450,280,120,50,26.666667
1,2,500,350,200,-50,40.000000
2,3,600,300,100,200,16.666667
3,4,400,310,150,-60,37.500000
4,5,700,400,250,50,35.714286


In [4]:

# Function to create financial instructions

def create_prompt(case):

    prompt = f"""
You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: ${case['income']}
Basic expenses: ${case['expenses']}
Monthly debt payments: ${case['debt_payments']}
Remaining balance: ${case['remaining_balance']:.2f}
Income allocated to debt: {case['debt_percentage']:.2f}%

Please provide:

1. A simple explanation of the financial situation.
2. Possible warning signs of over-indebtedness.
3. Two practical financial education recommendations.

Do not invent information.
Do not recommend taking out new loans.
Use simple and easy-to-understand language.

"""

    return prompt


# Test the function using the first financial case
test_prompt = create_prompt(df.iloc[0])

print(test_prompt)



You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: $450.0
Basic expenses: $280.0
Monthly debt payments: $120.0
Remaining balance: $50.00
Income allocated to debt: 26.67%

Please provide:

1. A simple explanation of the financial situation.
2. Possible warning signs of over-indebtedness.
3. Two practical financial education recommendations.

Do not invent information.
Do not recommend taking out new loans.
Use simple and easy-to-understand language.




In [5]:

# Authenticate with Hugging Face

from huggingface_hub import login

login()

print("Hugging Face authentication completed.")


Hugging Face authentication completed.


In [6]:

# Load the first AI model: Qwen2.5-0.5B-Instruct

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading Qwen model...")

# Load the tokenizer
tokenizer_qwen = AutoTokenizer.from_pretrained(model_name)

# Load the language model
model_qwen = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

# Set the model to evaluation mode
model_qwen.eval()

print("Qwen model loaded successfully!")


Loading Qwen model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen model loaded successfully!


In [7]:

# Generate the first financial response with Qwen

import time

# Prepare the conversation
messages = [
    {"role": "user", "content": test_prompt}
]

# Convert the prompt into model inputs
inputs = tokenizer_qwen.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

# Start measuring generation time
start_time = time.perf_counter()

# Generate the response
with torch.inference_mode():
    outputs = model_qwen.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer_qwen.eos_token_id
    )

# Calculate generation time
generation_time = time.perf_counter() - start_time

# Extract the generated tokens
generated_tokens = outputs[0][
    inputs["input_ids"].shape[1]:
]

# Decode the generated response
qwen_response = tokenizer_qwen.decode(
    generated_tokens,
    skip_special_tokens=True
)

# Display the results
print("QWEN FINANCIAL ANALYSIS:")
print(qwen_response)

print("\nGeneration time:", round(generation_time, 2), "seconds")


QWEN FINANCIAL ANALYSIS:
### Financial Situation Analysis

**1. Simple Explanation of the Financial Situation:**

The provided financial situation is as follows:
- **Monthly Income:** $450.00
- **Basic Expenses:** $280.00
- **Remaining Balance:** $50.00
- **Monthly Debt Payments:** $120.00
- **Income Allocated to Debt:** 26.67%

This means that after covering basic expenses and paying off some debt, there is still $50.00 left in the remaining balance.

### Possible Warning Signs of Over-Indebtedness

Over-indebtedness can be identified through several indicators:
- **Excessive Spending:** If you find yourself spending more than your income, it might indicate an over-indebted state.
- **Lack of Savings:** If you don't have

Generation time: 36.8 seconds


In [8]:

# Improve the financial analysis prompt

def create_prompt(case):
    prompt = f"""
You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: ${case['income']}
Basic expenses: ${case['expenses']}
Monthly debt payments: ${case['debt_payments']}
Remaining balance: ${case['remaining_balance']:.2f}
Income allocated to debt: {case['debt_percentage']:.2f}%

Provide a short financial analysis using this format:

1. Financial Situation:
Explain the person's financial situation in 2 sentences.

2. Financial Risk:
Identify possible warning signs of over-indebtedness
in 2 sentences.

3. Recommendations:
Provide exactly 2 practical financial recommendations.

Rules:
- Respond only in English.
- Use simple and clear language.
- Do not invent financial information.
- Do not recommend taking out new loans.
- Keep the response concise.
"""
    return prompt

# Update the test prompt
test_prompt = create_prompt(df.iloc[0])

print(test_prompt)



You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: $450.0
Basic expenses: $280.0
Monthly debt payments: $120.0
Remaining balance: $50.00
Income allocated to debt: 26.67%

Provide a short financial analysis using this format:

1. Financial Situation:
Explain the person's financial situation in 2 sentences.

2. Financial Risk:
Identify possible warning signs of over-indebtedness
in 2 sentences.

3. Recommendations:
Provide exactly 2 practical financial recommendations.

Rules:
- Respond only in English.
- Use simple and clear language.
- Do not invent financial information.
- Do not recommend taking out new loans.
- Keep the response concise.



In [9]:

# Generate an improved financial response with Qwen

import time

# Prepare the updated prompt
messages = [
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer_qwen.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

# Measure generation time
start_time = time.perf_counter()

with torch.inference_mode():
    outputs = model_qwen.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tokenizer_qwen.eos_token_id
    )

generation_time = time.perf_counter() - start_time

# Extract only the generated response
generated_tokens = outputs[0][
    inputs["input_ids"].shape[1]:
]

qwen_response = tokenizer_qwen.decode(
    generated_tokens,
    skip_special_tokens=True
)

# Save the results
qwen_results = {
    "model": "Qwen2.5-0.5B-Instruct",
    "case_id": 1,
    "response": qwen_response,
    "generation_time": generation_time,
    "generated_tokens": len(generated_tokens)
}

# Display the results
print("QWEN FINANCIAL ANALYSIS:")
print(qwen_response)

print("\nPERFORMANCE METRICS:")
print("Generation time:", round(generation_time, 2), "seconds")
print("Generated tokens:", len(generated_tokens))


QWEN FINANCIAL ANALYSIS:
1. **Financial Situation:**
   - The person has a basic monthly income of $450.00.
   - Their total monthly expenses include $280.00 for living costs and $120.00 for debts.
   - After deducting their debt payments from their income, they have a remaining balance of $50.00.
   - They allocate 26.67% of their income towards debt.

2. **Financial Risk:**
   - Over-indebtedness can lead to financial strain and potentially negative consequences such as increased interest rates on future debts or higher credit card balances.
   - It may also indicate that the person is struggling with managing their finances effectively.

3. **Recommendations:**
   - To mitigate potential risks associated with over-indebtedness, the person should consider reducing unnecessary expenses and increasing savings.
   - They could explore ways to reduce their debt payments (e.g., negotiating better terms with creditors) or look into alternative sources of income if feasible.
   - Additional

In [10]:

# Save the first Qwen evaluation result

import pandas as pd

qwen_evaluation = pd.DataFrame([
    {
        "model": qwen_results["model"],
        "case_id": qwen_results["case_id"],
        "generation_time": round(
            qwen_results["generation_time"], 2
        ),
        "generated_tokens": qwen_results["generated_tokens"],
        "tokens_per_second": round(
            qwen_results["generated_tokens"] /
            qwen_results["generation_time"], 2
        ),
        "response": qwen_results["response"]
    }
])

# Display the evaluation table
display(qwen_evaluation)


,model,case_id,generation_time,generated_tokens,tokens_per_second,response
0,Qwen2.5-0.5B-Instruct,1,49.55,242,4.88,1. **Financial Situation:**\n - The person h...


In [11]:

# Free memory before loading the second model

import gc
import torch

# Remove the Qwen model and temporary objects
del model_qwen
del tokenizer_qwen
del inputs
del outputs
del generated_tokens

# Run Python garbage collection
gc.collect()

# Verify that evaluation results are preserved
print("Qwen model removed from memory.")
print("Evaluation results preserved successfully.")

display(qwen_evaluation[
    [
        "model",
        "case_id",
        "generation_time",
        "generated_tokens",
        "tokens_per_second"
    ]
])


Qwen model removed from memory.
Evaluation results preserved successfully.


,model,case_id,generation_time,generated_tokens,tokens_per_second
0,Qwen2.5-0.5B-Instruct,1,49.55,242,4.88


In [12]:

# Download and load MNN-Pocket-0.3B

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name_mnn = "taobao-mnn/MNN-Pocket-0.3B"

print("Loading MNN-Pocket model...")

# Load the tokenizer
tokenizer_mnn = AutoTokenizer.from_pretrained(
    model_name_mnn
)

# Load the model on CPU
model_mnn = AutoModelForCausalLM.from_pretrained(
    model_name_mnn,
    dtype=torch.float32
)

# Set the model to evaluation mode
model_mnn.eval()

print("MNN-Pocket model loaded successfully!")


Loading MNN-Pocket model...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

MNN-Pocket model loaded successfully!


In [13]:

import pandas as pd

# Create fictional financial cases
cases = [
    {"id": 1, "income": 450, "expenses": 280, "debt_payments": 120},
    {"id": 2, "income": 500, "expenses": 350, "debt_payments": 200},
    {"id": 3, "income": 600, "expenses": 300, "debt_payments": 100},
    {"id": 4, "income": 400, "expenses": 310, "debt_payments": 150},
    {"id": 5, "income": 700, "expenses": 400, "debt_payments": 250}
]

# Convert the cases into a DataFrame
df = pd.DataFrame(cases)

# Calculate the remaining balance
df["remaining_balance"] = (
    df["income"] - df["expenses"] - df["debt_payments"]
)

# Calculate the percentage of income used for debt payments
df["debt_percentage"] = (
    df["debt_payments"] / df["income"] * 100
)

# Display the financial data
display(df)


,id,income,expenses,debt_payments,remaining_balance,debt_percentage
0,1,450,280,120,50,26.666667
1,2,500,350,200,-50,40.000000
2,3,600,300,100,200,16.666667
3,4,400,310,150,-60,37.500000
4,5,700,400,250,50,35.714286


In [14]:

# Improve the financial analysis prompt

def create_prompt(case):
    prompt = f"""
You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: ${case['income']}
Basic expenses: ${case['expenses']}
Monthly debt payments: ${case['debt_payments']}
Remaining balance: ${case['remaining_balance']:.2f}
Income allocated to debt: {case['debt_percentage']:.2f}%

Provide a short financial analysis using this format:

1. Financial Situation:
Explain the person's financial situation in 2 sentences.

2. Financial Risk:
Identify possible warning signs of over-indebtedness
in 2 sentences.

3. Recommendations:
Provide exactly 2 practical financial recommendations.

Rules:
- Respond only in English.
- Use simple and clear language.
- Do not invent financial information.
- Do not recommend taking out new loans.
- Keep the response concise.
"""
    return prompt

# Update the test prompt
test_prompt = create_prompt(df.iloc[0])

print(test_prompt)



You are a financial education assistant.

Analyze the following monthly financial situation:

Monthly income: $450.0
Basic expenses: $280.0
Monthly debt payments: $120.0
Remaining balance: $50.00
Income allocated to debt: 26.67%

Provide a short financial analysis using this format:

1. Financial Situation:
Explain the person's financial situation in 2 sentences.

2. Financial Risk:
Identify possible warning signs of over-indebtedness
in 2 sentences.

3. Recommendations:
Provide exactly 2 practical financial recommendations.

Rules:
- Respond only in English.
- Use simple and clear language.
- Do not invent financial information.
- Do not recommend taking out new loans.
- Keep the response concise.



In [15]:

# Generate the first financial response with MNN-Pocket

import time
import torch

# Prepare the same financial prompt used with Qwen
messages = [
    {"role": "user", "content": test_prompt}
]

inputs_mnn = tokenizer_mnn.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
)

# Measure generation time
start_time = time.perf_counter()

with torch.inference_mode():
    outputs_mnn = model_mnn.generate(
        **inputs_mnn,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tokenizer_mnn.eos_token_id
    )

generation_time_mnn = time.perf_counter() - start_time

# Extract generated tokens
generated_tokens_mnn = outputs_mnn[0][
    inputs_mnn["input_ids"].shape[1]:
]

# Decode the response
mnn_response = tokenizer_mnn.decode(
    generated_tokens_mnn,
    skip_special_tokens=True
)

# Save the evaluation results
mnn_results = {
    "model": "MNN-Pocket-0.3B",
    "case_id": 1,
    "response": mnn_response,
    "generation_time": generation_time_mnn,
    "generated_tokens": len(generated_tokens_mnn)
}

# Display the response and performance metrics
print("MNN-POCKET FINANCIAL ANALYSIS:")
print(mnn_response)

print("\nPERFORMANCE METRICS:")
print("Generation time:", round(generation_time_mnn, 2), "seconds")
print("Generated tokens:", len(generated_tokens_mnn))


MNN-POCKET FINANCIAL ANALYSIS:
1. Financial Situation: The person has a stable income of $450.0, but is facing basic expenses of $280.0, and has $120 in debt payments. The remaining balance is $50.00, which indicates a small gap between income and expenses.  

2. Financial Risk: Over-debt is a warning sign if the person is paying more than they can afford, especially if they are not using their income to cover debt.  

3. Recommendations:  
- Set a monthly debt limit of $150 to avoid overspending and ensure financial stability.  
- Allocate the remaining $50 to save for emergencies or future goals.

PERFORMANCE METRICS:
Generation time: 20.32 seconds
Generated tokens: 142


In [16]:

# Restore Qwen results from previously recorded measurements

import pandas as pd

qwen_evaluation = pd.DataFrame([
    {
        "model": "Qwen2.5-0.5B-Instruct",
        "case_id": 1,
        "generation_time": 49.78,
        "generated_tokens": 242,
        "tokens_per_second": round(242 / 49.78, 2),
        
    }
])

print("Qwen performance metrics restored successfully!")

display(qwen_evaluation)


Qwen performance metrics restored successfully!


,model,case_id,generation_time,generated_tokens,tokens_per_second
0,Qwen2.5-0.5B-Instruct,1,49.78,242,4.86


In [17]:

# Compare Qwen and MNN-Pocket performance

import pandas as pd

# Restore MNN-Pocket evaluation if its results are available
if "mnn_evaluation" not in globals():
    if "mnn_results" in globals():
        mnn_evaluation = pd.DataFrame([
            {
                "model": mnn_results["model"],
                "case_id": mnn_results["case_id"],
                "generation_time": round(
                    mnn_results["generation_time"], 2
                ),
                "generated_tokens": mnn_results["generated_tokens"],
                "tokens_per_second": round(
                    mnn_results["generated_tokens"] /
                    mnn_results["generation_time"], 2
                ),
                "response": mnn_results["response"]
            }
        ])
    else:
        # Previously recorded MNN-Pocket measurements
        mnn_evaluation = pd.DataFrame([
            {
                "model": "MNN-Pocket-0.3B",
                "case_id": 1,
                "generation_time": 21.20,
                "generated_tokens": 142,
                "tokens_per_second": round(142 / 21.20, 2),
                "response": None
            }
        ])

# Combine both evaluation tables
comparison_df = pd.concat(
    [qwen_evaluation, mnn_evaluation],
    ignore_index=True
)

# Display the performance comparison
display(
    comparison_df[
        [
            "model",
            "case_id",
            "generation_time",
            "generated_tokens",
            "tokens_per_second"
        ]
    ]
)

# Identify the fastest model
fastest_model = comparison_df.loc[
    comparison_df["generation_time"].idxmin(), "model"
]

print("Fastest model for case 1:", fastest_model)


,model,case_id,generation_time,generated_tokens,tokens_per_second
0,Qwen2.5-0.5B-Instruct,1,49.78,242,4.86
1,MNN-Pocket-0.3B,1,20.32,142,6.99


Fastest model for case 1: MNN-Pocket-0.3B
